# GameTheory-21 : Capstone — la sous-série SocialChoice en une heure

**Série** : GameTheory — jumeau Rust du carnet Python
**Source du portage** : [GameTheory-21-Capstone-Choix-Social-Python.ipynb](GameTheory-21-Capstone-Choix-Social-Python.ipynb)
**Kernel** : `rust` (evcxr, `std` seul) — mêmes témoins recalculés : census d'Arrow, cycle de Condorcet, Gale-Shapley, lecture du lake Lean

**Navigation** : [<< 20-Capstone-Coalitions](GameTheory-20-Capstone-Coalitions-Python.ipynb) | [Index](README.md) | [Sous-série SocialChoice >>](SocialChoice/README.md) | [Version Python](GameTheory-21-Capstone-Choix-Social-Python.ipynb)

**Side tracks** : ce carnet EST l'escalier vers la sous-série [SocialChoice](SocialChoice/README.md) — il la présente et n'en remplace aucun carnet.

Ce carnet est le **capstone** qui referme le parcours principal de la série GameTheory et ouvre sa sous-série SocialChoice, consacrée à l'agrégation des préférences. Il répond à trois questions, dans l'ordre où un lecteur du parcours les pose :

1. **Ce que la sous-série démontre** — les résultats qu'elle établit, et pourquoi ils comptent au-delà de la théorie des jeux ;
2. **Ce qu'elle suppose** — les prérequis, palier par palier, pour y entrer au bon endroit ;
3. **Par où y entrer** — trois parcours de lecture selon votre niveau.

Il le fait en **trois expériences de dix minutes** (une simulation du théorème d'Arrow, un cycle de Condorcet, un mariage stable de Gale-Shapley) et en **citant le lake Lean** de la sous-série, dont les théorèmes sont prouvés formellement — pas seulement simulés.

***

## Ce que ce notebook suppose

- **Prérequis** : le parcours [GameTheory 01-16](README.md), en particulier la [forme normale](GameTheory-02-NormalForm-Python.ipynb) (préférences ordonnées, notations) et le [palier 16](GameTheory-16-MechanismDesign-Python.ipynb) (le principe de révélation et l'appariement stable, dont Gale-Shapley est le moteur). Aucun prérequis en théorie du choix social : tout est introduit ici.
- **Outil** : Rust (noyau evcxr), bibliothèque standard uniquement. Le carnet s'exécute de bout en bout, exercices non complétés compris.
- **Public** : **Licence**. Les extensions vers la recherche (formalisation Lean, encodage SAT, dividendes de Harsanyi) sont citées et liées, pas supposées.

## 1. Ce que la sous-série démontre

La théorie du choix social étudie une question qui paraît simple : **comment transformer des préférences individuelles en une décision collective ?** Un vote, un classement, un comité, un appariement — dans chaque cas, une règle agrège des ordres individuels en un ordre (ou un choix) social.

La réponse de la sous-série tient en une famille de **théorèmes d'impossibilité** et en trois **changements d'objet** :

| Angle | Carnet | Ce qu'il établit | Durée | Public |
|---|---|---|---|---|
| Simulation d'Arrow | [SC-01](SocialChoice/ChoixSocial-01-Arrow-Impossibility-Theorem-Python.ipynb) | Aucune règle ne satisfait simultanément Pareto, IIA et non-dictature (preuve de Geanakoplos pas à pas, testée par le code) | 45 min | Licence |
| Preuve formelle | [SC-02](SocialChoice/ChoixSocial-01b-Lean-SocialChoice-Formal-Lean.ipynb) | Arrow, Sen et l'électeur médian **prouvés en Lean 4**, 0 `sorry` | 80 min | Recherche |
| Méthodes de vote | [SC-03](SocialChoice/ChoixSocial-03-Voting-Methods-Python.ipynb) | Condorcet, Borda, Copeland, Downs : chaque règle a son paradoxe | 35 min | Découverte |
| Vérification SAT | [SC-04](SocialChoice/ChoixSocial-04-Computational-Aggregation-SAT-Z3-Python.ipynb) | L'impossibilité devient un `UNSAT` : la machine vérifie tous les cas finis | 45 min | Licence |
| Manipulation | [SC-05](SocialChoice/ChoixSocial-05-Gibbard-Satterthwaite-Python.ipynb) | Gibbard-Satterthwaite : toute règle non triviale est manipulable — le témoin s'exhibe par le code | 30 min | Licence |
| Agrégation coopérative | [SC-06](SocialChoice/ChoixSocial-06-Mobius-Aggregation-Pouvoir-Manipulation-Rust.ipynb) | Möbius sur le treillis des coalitions : dividendes de Harsanyi, poids ≠ pouvoir | 40 min | Recherche |
| Comités | [SC-07](SocialChoice/ChoixSocial-07-Committees-Core-Python.ipynb) | Élire un comité : le core existe toujours (quotas Hare/Droop, certificats) | 40 min | Recherche |
| Mariage stable | [SC-08](SocialChoice/ChoixSocial-02-Stable-Marriage-Gale-Shapley-Rust.ipynb) | Gale-Shapley : couplage parfait, stabilité, man-optimalité — jumeau d'un lake Lean 0 `sorry` | 35 min | Licence |

Les trois impossibilités fondatrices (Arrow 1951, Sen 1970, Gibbard-Satterthwaite 1973) disent la même chose trois fois, pour trois objets : **un ordre social, une liberté individuelle, une règle de vote** — sous des axiomes « raisonnables », quelque chose doit casser. Les trois changements d'objet (SC-06 : agréger des valeurs de coalition ; SC-07 : élire un comité ; SC-08 : apparier deux populations) montent ce qui reste possible quand on change la question.

Les trois expériences qui suivent mettent chaque fois **un seul résultat en main**, dans le formalisme minimal pour le voir.


## 2. Trois expériences de dix minutes


### 2.1 Le théorème d'Arrow par la simulation

Trois axiomes, trois règles, un verdict. Les axiomes :

- **Pareto (faible)** : si *tous* les électeurs préfèrent `x` à `y`, la règle doit classer `x` avant `y` ;
- **IIA** (indépendance des alternatives non pertinentes) : le classement social de `x` vs `y` ne doit dépendre que des classements individuels de `x` vs `y` — pas des positions d'une troisième alternative `z` ;
- **Non-dictature** : aucun électeur seul ne impose son classement.

Le théorème d'Arrow (1951) : avec **3 alternatives ou plus**, aucune fonction de bien-être social ne satisfait les trois. La cellule suivante définit le matériel (profils et règles), la suivante teste les axiomes.


In [2]:
// Materiau commun aux experiences : un profil = liste d'ordres individuels.
// Convention : chaque ordre est une liste des alternatives, de la meilleure a la pire.

type Scores = Vec<(String, i64)>;

const ALTS: [&str; 3] = ["A", "B", "C"];

fn pos(o: &[&str], x: &str) -> usize {
    o.iter().position(|e| *e == x).unwrap()
}

fn incremente(scores: &mut Scores, alt: &str, v: i64) {
    for e in scores.iter_mut() {
        if e.0 == alt {
            e.1 += v;
            return;
        }
    }
}

// Score de Borda : chaque electeur donne m-1, m-2, ..., 0 points a ses alternatives.
fn borda(profil: &[Vec<&str>], alts: &[&str]) -> Scores {
    let m = alts.len() as i64;
    let mut scores: Scores = alts.iter().map(|a| (a.to_string(), 0)).collect();
    for ordre in profil {
        for (rang, alt) in ordre.iter().enumerate() {
            incremente(&mut scores, alt, m - 1 - rang as i64);
        }
    }
    scores
}

// Score de pluralite : 1 point a l'alternative classee premiere par chaque electeur.
fn pluralite(profil: &[Vec<&str>], alts: &[&str]) -> Scores {
    let mut scores: Scores = alts.iter().map(|a| (a.to_string(), 0)).collect();
    for ordre in profil {
        incremente(&mut scores, ordre[0], 1);
    }
    scores
}

// Score du dictateur k : son ordre personnel devient l'ordre social.
fn dictateur(k: usize, profil: &[Vec<&str>], alts: &[&str]) -> Scores {
    let m = alts.len() as i64;
    profil[k]
        .iter()
        .enumerate()
        .map(|(rang, alt)| (alt.to_string(), m - 1 - rang as i64))
        .collect()
}

// Ordre social decroissant selon les scores (lesegalites restent visibles).
fn ordre_social(scores: &Scores) -> Vec<String> {
    let mut v = scores.clone();
    v.sort_by(|a, b| b.1.cmp(&a.1));
    v.into_iter().map(|(k, _)| k).collect()
}

{
    println!("Regles definies : borda, pluralite, dictateur.");
};

Regles definies : borda, pluralite, dictateur.


In [3]:
// Test des trois axiomes sur les trois regles.
// Pareto : pour toute paire unanime x > y, l'ordre social doit placer x avant y.
// IIA : deux profils P1/P2 ou CHAQUE electeur garde le meme ordre entre A et B,
//       mais ou l'ordre social A vs B bascule -> violation.
// Non-dictature : la regle ne doit pas coincider systematiquement avec l'ordre d'un electeur.

type Scores = Vec<(String, i64)>;

const ALTS: [&str; 3] = ["A", "B", "C"];

fn pos(o: &[&str], x: &str) -> usize {
    o.iter().position(|e| *e == x).unwrap()
}

fn py_dict_i(scores: &Scores) -> String {
    let items: Vec<String> = scores.iter().map(|(k, v)| format!("'{}': {}", k, v)).collect();
    format!("{{{}}}", items.join(", "))
}

fn py_list_str(v: &[String]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}

fn incremente(scores: &mut Scores, alt: &str, v: i64) {
    for e in scores.iter_mut() {
        if e.0 == alt {
            e.1 += v;
            return;
        }
    }
}

// Score de Borda : chaque electeur donne m-1, m-2, ..., 0 points a ses alternatives.
fn borda(profil: &[Vec<&str>], alts: &[&str]) -> Scores {
    let m = alts.len() as i64;
    let mut scores: Scores = alts.iter().map(|a| (a.to_string(), 0)).collect();
    for ordre in profil {
        for (rang, alt) in ordre.iter().enumerate() {
            incremente(&mut scores, alt, m - 1 - rang as i64);
        }
    }
    scores
}

// Score de pluralite : 1 point a l'alternative classee premiere par chaque electeur.
fn pluralite(profil: &[Vec<&str>], alts: &[&str]) -> Scores {
    let mut scores: Scores = alts.iter().map(|a| (a.to_string(), 0)).collect();
    for ordre in profil {
        incremente(&mut scores, ordre[0], 1);
    }
    scores
}

// Score du dictateur k : son ordre personnel devient l'ordre social.
fn dictateur(k: usize, profil: &[Vec<&str>], alts: &[&str]) -> Scores {
    let m = alts.len() as i64;
    profil[k]
        .iter()
        .enumerate()
        .map(|(rang, alt)| (alt.to_string(), m - 1 - rang as i64))
        .collect()
}

// Ordre social decroissant selon les scores (lesegalites restent visibles).
fn ordre_social(scores: &Scores) -> Vec<String> {
    let mut v = scores.clone();
    v.sort_by(|a, b| b.1.cmp(&a.1));
    v.into_iter().map(|(k, _)| k).collect()
}

fn valeur(scores: &Scores, cle: &str) -> i64 {
    scores.iter().find(|(k, _)| k == cle).map(|(_, v)| *v).unwrap()
}

type Regle = fn(&[Vec<&str>], &[&str]) -> Scores;

fn pareto_ok(scores: &Scores, profil: &[Vec<&str>], alts: &[&str]) -> bool {
    let ordre = ordre_social(scores);
    let pos_de = |a: &str| ordre.iter().position(|e| e == a).unwrap();
    for i in 0..alts.len() {
        for y in &alts[i + 1..] {
            let x = alts[i];
            // Unanimite x > y : l'ordre social doit placer x avant y...
            if profil.iter().all(|o| pos(o, x) < pos(o, *y)) && !(pos_de(x) < pos_de(*y)) {
                return false;
            }
            // ...et unanimite y > x tout autant : chaque paire a DEUX orientations,
            // et l'unanimite inverse est un cas Pareto a part entiere.
            if profil.iter().all(|o| pos(o, *y) < pos(o, x)) && !(pos_de(*y) < pos_de(x)) {
                return false;
            }
        }
    }
    true
}

// True si le temoin ne bascule PAS (IIA respectee sur ce cas), false s'il
// bascule, None si la premisse est invalide : la premisse IIA exige que CHAQUE
// electeur garde son ordre x-vs-y entre P1 et P2 -- vecteur individuel par
// vecteur individuel, pas leur seul `all(...)`. Un couple dont la premisse ne
// tient pas ne prouve rien sur la regle.
fn iia_temoin(f: Regle, p1: &[Vec<&str>], p2: &[Vec<&str>], alts: &[&str], x: &str, y: &str) -> Option<bool> {
    let prefs1: Vec<bool> = p1.iter().map(|o| pos(o, x) < pos(o, y)).collect();
    let prefs2: Vec<bool> = p2.iter().map(|o| pos(o, x) < pos(o, y)).collect();
    if prefs1 != prefs2 {
        return None;
    }
    let soc = |p: &[Vec<&str>]| valeur(&f(p, alts), x) > valeur(&f(p, alts), y);
    Some(soc(p1) == soc(p2))
}

{
    // Temoins IIA (mesures ci-dessous) :
    //  pluralite : P1 -> social A>B ; P2 -> social B>A (preferences A-vs-B individuelles identiques)
    //  borda     : P1' -> social B>A ; P2' -> social A>B (idem, 5 electeurs)
    let p1_plu: Vec<Vec<&str>> = vec![vec!["A", "B", "C"], vec!["B", "C", "A"], vec!["A", "B", "C"]];
    let p2_plu: Vec<Vec<&str>> = vec![vec!["C", "A", "B"], vec!["B", "C", "A"], vec!["C", "A", "B"]];
    let p1_bor: Vec<Vec<&str>> = vec![
        vec!["A", "B", "C"], vec!["A", "B", "C"], vec!["A", "B", "C"],
        vec!["B", "C", "A"], vec!["B", "C", "A"],
    ];
    let p2_bor: Vec<Vec<&str>> = vec![
        vec!["A", "C", "B"], vec!["A", "C", "B"], vec!["A", "C", "B"],
        vec!["B", "A", "C"], vec!["B", "A", "C"],
    ];

    let profil_arrow: Vec<Vec<&str>> = vec![vec!["A", "B", "C"], vec!["B", "A", "C"], vec!["A", "C", "B"]];

    let mut lignes: Vec<(&str, bool, Option<bool>, bool)> = Vec::new();
    lignes.push(("Borda", pareto_ok(&borda(&profil_arrow, &ALTS), &profil_arrow, &ALTS), iia_temoin(borda, &p1_bor, &p2_bor, &ALTS, "A", "B"), true));
    lignes.push(("Pluralite", pareto_ok(&pluralite(&profil_arrow, &ALTS), &profil_arrow, &ALTS), iia_temoin(pluralite, &p1_plu, &p2_plu, &ALTS, "A", "B"), true));
    // Dictateur 0 : Pareto et IIA tiennent par construction, la non-dictature tombe.
    lignes.push(("Dictateur(v1)", pareto_ok(&dictateur(0, &profil_arrow, &ALTS), &profil_arrow, &ALTS), Some(true), false));

    let ok_viole = |b: bool| if b { "OK" } else { "VIOLE" };
    let iia_txt = |i: &Option<bool>| match i {
        Some(true) => "OK",
        Some(false) => "VIOLE",
        None => "N/A",
    };
    println!("{:<14} {:<8} {:<6} {:<14} Verdict", "Regle", "Pareto", "IIA", "Non-dictature");
    for (nom, p, i, nd) in &lignes {
        let verts = (*p as usize) + ((*i == Some(true)) as usize) + (*nd as usize);
        let suite = if verts < 3 { "-> impossible" } else { "" };
        println!("{:<14} {:<8} {:<6} {:<14} {}/3 {}", nom, ok_viole(*p), iia_txt(i), ok_viole(*nd), verts, suite);
    }
    println!();
    println!("Details des temoins IIA :");
    println!("  Pluralite : P1 scores {} -> ordre {}", py_dict_i(&pluralite(&p1_plu, &ALTS)), py_list_str(&ordre_social(&pluralite(&p1_plu, &ALTS))));
    println!("              P2 scores {} -> ordre {}", py_dict_i(&pluralite(&p2_plu, &ALTS)), py_list_str(&ordre_social(&pluralite(&p2_plu, &ALTS))));
    println!("  Borda     : P1' scores {} -> ordre {}", py_dict_i(&borda(&p1_bor, &ALTS)), py_list_str(&ordre_social(&borda(&p1_bor, &ALTS))));
    println!("              P2' scores {} -> ordre {}", py_dict_i(&borda(&p2_bor, &ALTS)), py_list_str(&ordre_social(&borda(&p2_bor, &ALTS))));

    // Controles de non-regression : les contre-exemples de la review (#20663),
    // figes ici pour qu'aucune reecriture des predicats ne les reperde.
    // 1. Unanimite INVERSE : tous les electeurs preferent C>B>A pendant que
    //    l'ordre social affiche A>B>C -- Pareto doit tomber.
    let faux_scores: Scores = vec![("A".to_string(), 3), ("B".to_string(), 2), ("C".to_string(), 1)];
    let unanime_cba: Vec<Vec<&str>> = vec![vec!["C", "B", "A"]; 2];
    let unanime_abc: Vec<Vec<&str>> = vec![vec!["A", "B", "C"]; 2];
    assert!(!pareto_ok(&faux_scores, &unanime_cba, &ALTS));
    // 2. Controle positif : le meme ordre social face a l'unanimite directe tient.
    assert!(pareto_ok(&faux_scores, &unanime_abc, &ALTS));
    // 3. Premisse IIA invalide : les vecteurs individuels A-vs-B de P et Q
    //    different electeur par electeur -- le couple ne prouve rien.
    let p_inv: Vec<Vec<&str>> = vec![vec!["A", "B", "C"], vec!["B", "A", "C"]];
    let q_inv: Vec<Vec<&str>> = vec![vec!["B", "A", "C"], vec!["A", "B", "C"]];
    assert!(iia_temoin(borda, &p_inv, &q_inv, &ALTS, "A", "B").is_none());
    // 4-5. Les temoins concrets du tableau gardent une premisse valide : chaque
    //      electeur conserve bien son ordre A-vs-B entre P1 et P2.
    let v1: Vec<bool> = p1_plu.iter().map(|o| pos(o, "A") < pos(o, "B")).collect();
    let v2: Vec<bool> = p2_plu.iter().map(|o| pos(o, "A") < pos(o, "B")).collect();
    assert!(v1 == v2);
    let w1: Vec<bool> = p1_bor.iter().map(|o| pos(o, "A") < pos(o, "B")).collect();
    let w2: Vec<bool> = p2_bor.iter().map(|o| pos(o, "A") < pos(o, "B")).collect();
    assert!(w1 == w2);
    println!();
    println!("Controles de non-regression (contre-exemples review #20663) : 5/5 OK");
};

Regle          Pareto   IIA    Non-dictature  Verdict


Borda          OK       VIOLE  OK             2/3 -> impossible


Pluralite      OK       VIOLE  OK             2/3 -> impossible


Dictateur(v1)  OK       OK     VIOLE          2/3 -> impossible


Details des temoins IIA :


  Pluralite : P1 scores {'A': 2, 'B': 1, 'C': 0} -> ordre ['A', 'B', 'C']


              P2 scores {'A': 0, 'B': 1, 'C': 2} -> ordre ['C', 'B', 'A']


  Borda     : P1' scores {'A': 6, 'B': 7, 'C': 2} -> ordre ['B', 'A', 'C']


              P2' scores {'A': 8, 'B': 4, 'C': 3} -> ordre ['A', 'B', 'C']


Controles de non-regression (contre-exemples review #20663) : 5/5 OK


**Lecture du résultat.** Aucune ligne n'est verte des trois colonnes : Borda et la pluralité violent l'IIA (déplacer **C** — une alternative perdante — suffit à inverser le classement social de A vs B), la dictature viole la non-dictature. C'est le théorème d'Arrow **vu par le code** : chaque axiome isolé est satisfaisable, la conjonction ne l'est pas — sur ces exemples.

Ce que la simulation **ne montre pas** : elle teste trois règles sur un profil. Arrow dit que **toutes** les règles (avec 3+ alternatives) échouent — une infinité de cas que seul un raisonnement couvre. SC-01 déroule la preuve de Geanakoplos pas à pas ; SC-04 la fait vérifier par un solveur SAT sur les profils à 3 électeurs ; le lake Lean (section 3 ci-dessous) la prouve une fois pour toutes.

Le témoin qui suit ferme cette porte : sur **3 alternatives**, la famille des règles IIA + Pareto est **finie**, et son recensement est exhaustif.

### 2.1.1 — Recenser la famille : combien de règles IIA + Pareto existe-t-il ?

La simulation ci-dessus teste **trois** règles sur **quelques** profils. La question qu'elle laisse ouverte — « une *autre* règle aurait-elle survécu ? » — porte sur une **famille**, pas sur un échantillon.

Pour **3 alternatives** et **3 électeurs**, cette famille est **finie**, donc énumérable :

- l'**IIA** impose que l'ordre social sur une paire `{x, y}` ne dépende que du **motif** des préférences individuelles sur cette paire (le masque des électeurs qui préférez `x` à `y`) ;
- le **Pareto** impose le verdict sur les deux motifs unanimes (tout le monde préfère `x` : `x` l'emporte ; personne ne préfère `x` : `y` l'emporte) ;
- il reste donc **6 motifs libres** par paire — les six motifs non unanimes sur trois électeurs — soit `2^6 = 64` fonctions de décision par paire, et `64^3 = 262 144` règles candidates au total.

Le témoin qui suit énumère ces **262 144** candidates et ne retient que celles qui produisent un **ordre total strict** sur les `6^3 = 216` profils possibles. Le résultat est **déterministe** : aucune donnée aléatoire n'intervient.


In [4]:
// Recensement exhaustif de la famille des regles IIA + Pareto, sur 3 alternatives
// et 3 electeurs. Le resultat est deterministe : aucune donnee aleatoire.

const N_VOT: usize = 3;

fn pos(o: &[&str], x: &str) -> usize {
    o.iter().position(|e| *e == x).unwrap()
}

// Les 64 fonctions de decision d'une paire : motif -> vainqueur.
// Pareto fixe les deux motifs unanimes (masques 0 et 7) ; les six autres sont
// libres, d ou 2^6 = 64 fonctions.
fn fonctions_de_paire(x: &'static str, y: &'static str) -> Vec<[&'static str; 8]> {
    let mut fonctions = Vec::new();
    for bits in 0..64u32 {
        let mut table: [&'static str; 8] = [""; 8];
        for masque in 0..8usize {
            table[masque] = if masque == 0 {
                y
            } else if masque == 7 {
                x
            } else if (bits >> (masque - 1)) & 1 == 0 {
                x
            } else {
                y
            };
        }
        fonctions.push(table);
    }
    fonctions
}

// Vrai si les vainqueurs des trois paires forment un ordre strict (pas de cycle).
fn est_ordre_total(arcs: &[(&'static str, &'static str); 3]) -> bool {
    for (a, b) in arcs.iter() {
        for (c, d) in arcs.iter() {
            for (e, f) in arcs.iter() {
                if b == c && d == e && a == f {
                    return false;
                }
            }
        }
    }
    true
}

// Rang social (1 = premier) des alternatives, ou None si le verdict cycle.
// Noms locaux a la cellule : la signature differe de l'ordre_social des cellules 4-5.
fn ordre_social_census(
    fonctions: &[[&'static str; 8]; 3],
    motifs: &[u8; 3],
    paires: &[(&'static str, &'static str); 3],
    alts_c: &[&'static str; 3],
) -> Option<[i64; 3]> {
    let mut arcs: [(&'static str, &'static str); 3] = [("", ""); 3];
    for i in 0..3 {
        let (x, y) = paires[i];
        let vainqueur = fonctions[i][motifs[i] as usize];
        arcs[i] = (vainqueur, if vainqueur == x { y } else { x });
    }
    if !est_ordre_total(&arcs) {
        return None;
    }
    let mut rangs = [0i64; 3];
    for (j, a) in alts_c.iter().enumerate() {
        rangs[j] = 1 + arcs.iter().filter(|(_, perdant)| perdant == a).count() as i64;
    }
    Some(rangs)
}

// Index de l'electeur dont l'ordre personnel coincide toujours avec l'ordre social.
fn electeur_dictateur(
    fonctions: &[[&'static str; 8]; 3],
    profils: &[Vec<[&'static str; 3]>],
    motifs: &[[u8; 3]],
    paires: &[(&'static str, &'static str); 3],
) -> Option<usize> {
    for v in 0..N_VOT {
        let mut dicte = true;
        'ext: for (prof, m) in profils.iter().zip(motifs.iter()) {
            for i in 0..3 {
                let (x, y) = paires[i];
                let vainqueur = fonctions[i][m[i] as usize];
                let attendu = if pos(&prof[v], x) < pos(&prof[v], y) { x } else { y };
                if vainqueur != attendu {
                    dicte = false;
                    break 'ext;
                }
            }
        }
        if dicte {
            return Some(v);
        }
    }
    None
}

// Equivalent du format Python {:,} puis replace(",", " ").
fn groupe(n: usize) -> String {
    let s = n.to_string();
    let bytes = s.as_bytes();
    let mut out = String::new();
    for (i, c) in bytes.iter().enumerate() {
        if i > 0 && (bytes.len() - i) % 3 == 0 {
            out.push(' ');
        }
        out.push(*c as char);
    }
    out
}

{
    let alts_c: [&'static str; 3] = ["A", "B", "C"];
    let paires: [(&'static str, &'static str); 3] = [("A", "B"), ("A", "C"), ("B", "C")];
    // product(permutations(ALTS_C), repeat=3) : le dernier facteur varie le plus vite.
    let perms: Vec<[&'static str; 3]> = vec![
        ["A", "B", "C"], ["A", "C", "B"], ["B", "A", "C"],
        ["B", "C", "A"], ["C", "A", "B"], ["C", "B", "A"],
    ];
    let mut profils: Vec<Vec<[&'static str; 3]>> = Vec::new();
    for p0 in &perms {
        for p1 in &perms {
            for p2 in &perms {
                profils.push(vec![*p0, *p1, *p2]);
            }
        }
    }
    // Pour chaque profil, le motif des electeurs qui preference x a y, pour chaque paire.
    let motifs: Vec<[u8; 3]> = profils
        .iter()
        .map(|prof| {
            let mut m = [0u8; 3];
            for (i, (x, y)) in paires.iter().enumerate() {
                for (vi, o) in prof.iter().enumerate() {
                    if pos(o, x) < pos(o, y) {
                        m[i] += 1u8 << vi;
                    }
                }
            }
            m
        })
        .collect();

    let candidats = 64usize * 64 * 64;
    let tables: Vec<Vec<[&'static str; 8]>> =
        paires.iter().map(|(x, y)| fonctions_de_paire(x, y)).collect();
    let mut survivants: Vec<[[&'static str; 8]; 3]> = Vec::new();
    for f1 in &tables[0] {
        for f2 in &tables[1] {
            for f3 in &tables[2] {
                let fonctions = [*f1, *f2, *f3];
                if motifs
                    .iter()
                    .all(|m| ordre_social_census(&fonctions, m, &paires, &alts_c).is_some())
                {
                    survivants.push(fonctions);
                }
            }
        }
    }

    println!("Candidats IIA + Pareto enumeres : {} (64 fonctions x 3 paires)", groupe(candidats));
    println!("Profils de 3 electeurs sur 3 alternatives : {}", profils.len());
    println!("Survivants (ordre total strict sur les {} profils) : {}", profils.len(), survivants.len());

    for (k, fonctions) in survivants.iter().enumerate() {
        // Noms locaux a la cellule : ici le dictateur est un entier, pas une regle.
        let dictateur = electeur_dictateur(fonctions, &profils, &motifs, &paires);
        match dictateur {
            Some(d) => println!("  survivant {} : electeur dictateur n° {}", k + 1, d + 1),
            None => println!("  survivant {} : aucun electeur dictateur", k + 1),
        }
    }

    let ecartes = candidats - survivants.len();
    println!("Regles ecartees (ordre social non total sur un profil) : {}", groupe(ecartes));
    println!("Verdict : les survivants sont exactement les dictatures -- aucune des regles");
    println!("recensees ne satisfait la non-dictature. L'impossibilite d'Arrow est donc ENUMERABLE");
    println!("sur 3 alternatives ; le census est la version executable du raisonnement de SC-01.");
};

Candidats IIA + Pareto enumeres : 262 144 (64 fonctions x 3 paires)


Profils de 3 electeurs sur 3 alternatives : 216


Survivants (ordre total strict sur les 216 profils) : 3


  survivant 1 : electeur dictateur n° 3


  survivant 2 : electeur dictateur n° 2


  survivant 3 : electeur dictateur n° 1


Regles ecartees (ordre social non total sur un profil) : 262 141


Verdict : les survivants sont exactement les dictatures -- aucune des regles


recensees ne satisfait la non-dictature. L'impossibilite d'Arrow est donc ENUMERABLE


sur 3 alternatives ; le census est la version executable du raisonnement de SC-01.


**Lecture du résultat.** Le recensement est **exhaustif** : sur les **262 144** règles candidates (64 fonctions de décision par paire, sur trois paires), **exactement 3** produisent un ordre social total sur les **216** profils — et ce sont **les trois dictatures**, une par électeur. Les **262 141** autres sont écartées parce qu'elles cyclent sur au moins un profil.

Ce que cela ajoute à la simulation du début : parmi les règles recensées, **aucune ne satisfait la non-dictature** — et pas seulement « les trois règles testées échouent ». Sur **3 alternatives et 3 électeurs**, l'impossibilité d'Arrow (*Social Choice and Individual Values*, Cowles Commission Monograph No. 12, John Wiley & Sons, 1951) est **finie et énumérable** : la contradiction se lit dans une liste de 262 144 candidats dont 3 survivent, tous dictatoriaux.

Ce que le témoin **ne** montre pas : il vaut pour **3 alternatives et 3 électeurs**. Dès **4 alternatives**, le nombre de profils (`24^m`) rend l'énumération directe impraticable et la famille des règles trop grande pour un recensement à la main — c'est là que la **preuve** de SC-01 (argument du pivot) et le **solveur** de SC-04 prennent le relais. Le census est le **cas fini, vérifiable à la main** ; il ne remplace ni SC-01 ni SC-04, il en montre le point d'entrée.


### 2.2 Le cycle de Condorcet

Le marquis de Condorcet a proposé en 1785 un critère qui semble inattaquable : si une alternative bat toutes les autres en duel majoritaire, c'est elle qu'il faut élire. Le profil ci-dessous — le plus petit profil paradoxaux — montre pourquoi ce critère peut être **vide** : personne ne bat tout le monde.


In [5]:
// Le profil cyclique minimal : 3 electeurs, 3 alternatives.

const ALTS: [&str; 3] = ["A", "B", "C"];

fn pos(o: &[&str], x: &str) -> usize {
    o.iter().position(|e| *e == x).unwrap()
}

fn py_dict_i(scores: &[(String, i64)]) -> String {
    let items: Vec<String> = scores.iter().map(|(k, v)| format!("'{}': {}", k, v)).collect();
    format!("{{{}}}", items.join(", "))
}

fn incremente(scores: &mut Vec<(String, i64)>, alt: &str, v: i64) {
    for e in scores.iter_mut() {
        if e.0 == alt {
            e.1 += v;
            return;
        }
    }
}

// Score de Borda : chaque electeur donne m-1, m-2, ..., 0 points a ses alternatives.
fn borda(profil: &[Vec<&str>], alts: &[&str]) -> Vec<(String, i64)> {
    let m = alts.len() as i64;
    let mut scores: Vec<(String, i64)> = alts.iter().map(|a| (a.to_string(), 0)).collect();
    for ordre in profil {
        for (rang, alt) in ordre.iter().enumerate() {
            incremente(&mut scores, alt, m - 1 - rang as i64);
        }
    }
    scores
}

// Score de pluralite : 1 point a l'alternative classee premiere par chaque electeur.
fn pluralite(profil: &[Vec<&str>], alts: &[&str]) -> Vec<(String, i64)> {
    let mut scores: Vec<(String, i64)> = alts.iter().map(|a| (a.to_string(), 0)).collect();
    for ordre in profil {
        incremente(&mut scores, ordre[0], 1);
    }
    scores
}

// Pour chaque paire, qui gagne au vote majoritaire (et de combien) ?
fn duels(profil: &[Vec<&str>], alts: &[&str]) {
    for i in 0..alts.len() {
        for y in &alts[i + 1..] {
            let x = alts[i];
            let pour_x = profil.iter().filter(|o| pos(o, x) < pos(o, *y)).count();
            let pour_y = profil.len() - pour_x;
            let vainqueur = if pour_x > pour_y {
                x
            } else if pour_y > pour_x {
                y
            } else {
                "egalite"
            };
            println!("  {} vs {} : {}-{} -> {}", x, y, pour_x, pour_y, vainqueur);
        }
    }
}

{
    let cycle: Vec<Vec<&str>> = vec![vec!["A", "B", "C"], vec!["B", "C", "A"], vec!["C", "A", "B"]];
    println!("Duels majoritaires sur le profil cyclique :");
    duels(&cycle, &ALTS);
    println!();
    println!("Scores de Borda sur le meme profil : {}", py_dict_i(&borda(&cycle, &ALTS)));
    println!("Scores de pluralite : {}", py_dict_i(&pluralite(&cycle, &ALTS)));
    println!("Scores de Copeland (1 victoire, 0.5 nul, 0 defaite) : a vous -- Exercice 1.");
};

Duels majoritaires sur le profil cyclique :


  A vs B : 2-1 -> A


  A vs C : 1-2 -> C


  B vs C : 2-1 -> B


Scores de Borda sur le meme profil : {'A': 3, 'B': 3, 'C': 3}


Scores de pluralite : {'A': 1, 'B': 1, 'C': 1}


Scores de Copeland (1 victoire, 0.5 nul, 0 defaite) : a vous -- Exercice 1.


**Lecture du résultat.** A bat B (2-1), B bat C (2-1), C bat A (2-1) : le duel majoritaire **cycle**, il n'existe pas de gagnant de Condorcet. Pire : Borda et la pluralité rendent toutes deux une **égalité parfaite** (Borda 3/3/3, pluralité 1/1/1 — une voix chacune) — les deux règles sont muettes exactement là où le critère de Condorcet est vide.

Ce n'est pas un accident de fabrication : le théorème de birds-of-a-feather n'existe pas, mais le **paradoxe d'Arrow** (ci-dessus) et ce cycle sont les deux faces d'une même rigidité — agréger des ordres *sans jamais perdre d'information* est impossible, chaque règle choisit quelle information elle sacrifie. SC-03 déroule le bestiaire complet (Condorcet, Borda, Copeland, Dowd, paradoxe de l'échange, électeur médian) et montre quel paradoxe chaque règle accepte.


### 2.3 Mariage stable : Gale-Shapley

Troisième changement d'objet : on n'élit plus, on **apparie**. Deux populations (par convention des « proposants » et des « disposantes ») expriment chacune un ordre de préférence sur l'autre. Un appariement est **stable** si aucune paire (un proposant, une disposante) ne préfère mutuellement quitter leurs partenaires respectifs pour se mettre ensemble — une telle paire est dite **bloquante**.

L'algorithme de Gale-Shapley (1962) : les proposants demandent, les disposantes gardent provisoirement le meilleur et refusent le reste, jusqu'à ce que plus personne ne demande. Il produit toujours un appariement stable — et il est **optimal pour les proposants** : aucun proposant ne peut faire mieux dans aucun appariement stable.


In [6]:
// Gale-Shapley sur 4x4 preferences fixes (deterministe).

fn py_dict_ss(paires: &[(&str, &str)]) -> String {
    let items: Vec<String> = paires.iter().map(|(k, v)| format!("'{}': '{}'", k, v)).collect();
    format!("{{{}}}", items.join(", "))
}

// Deferred acceptance : les proposants demandent, les disposantes choisissent.
// Ordre d'insertion partout : file des libres dans l'ordre des proposants,
// retour dans l'ordre des disposantes.
fn gale_shapley<'a>(
    proposants: &[(&'a str, [&'a str; 4])],
    disposantes: &[(&'a str, [&'a str; 4])],
) -> (Vec<(&'a str, &'a str)>, usize) {
    let mut libres: std::collections::VecDeque<usize> = (0..proposants.len()).collect();
    let mut prochain_choix = vec![0usize; proposants.len()];
    let mut fiance: Vec<Option<usize>> = vec![None; disposantes.len()];
    let mut demandes = 0usize;
    while let Some(pi) = libres.pop_front() {
        let d = proposants[pi].1[prochain_choix[pi]];
        prochain_choix[pi] += 1;
        demandes += 1;
        let di = disposantes.iter().position(|(n, _)| *n == d).unwrap();
        match fiance[di] {
            None => fiance[di] = Some(pi),
            Some(cur) => {
                let prefs = &disposantes[di].1;
                if prefs.iter().position(|h| *h == proposants[pi].0).unwrap()
                    < prefs.iter().position(|h| *h == proposants[cur].0).unwrap()
                {
                    libres.push_back(cur);
                    fiance[di] = Some(pi);
                } else {
                    libres.push_back(pi);
                }
            }
        }
    }
    let paires = fiance
        .iter()
        .enumerate()
        .map(|(di, opt)| (proposants[opt.unwrap()].0, disposantes[di].0))
        .collect();
    (paires, demandes)
}

{
    let proposants: [(&str, [&str; 4]); 4] = [
        ("h1", ["f1", "f2", "f3", "f4"]),
        ("h2", ["f2", "f1", "f4", "f3"]),
        ("h3", ["f3", "f4", "f1", "f2"]),
        ("h4", ["f4", "f3", "f2", "f1"]),
    ];
    let disposantes: [(&str, [&str; 4]); 4] = [
        ("f1", ["h2", "h1", "h4", "h3"]),
        ("f2", ["h1", "h2", "h3", "h4"]),
        ("f3", ["h4", "h3", "h2", "h1"]),
        ("f4", ["h3", "h4", "h1", "h2"]),
    ];
    let (m_hommes, nb_demandes) = gale_shapley(&proposants, &disposantes);
    let (m_femmes_brut, _) = gale_shapley(&disposantes, &proposants); // {disposante: proposant}
    let m_femmes: Vec<(&str, &str)> = m_femmes_brut.iter().map(|(f, h)| (*h, *f)).collect(); // rebase en {proposant: disposante}
    let mut t_hommes = m_hommes.clone();
    t_hommes.sort();
    let mut t_femmes = m_femmes.clone();
    t_femmes.sort();
    println!("Appariement optimal pour les proposants : {} ({} demandes)", py_dict_ss(&t_hommes), nb_demandes);
    println!("Appariement optimal pour les disposantes : {}", py_dict_ss(&t_femmes));
};

Appariement optimal pour les proposants : {'h1': 'f1', 'h2': 'f2', 'h3': 'f3', 'h4': 'f4'} (4 demandes)


Appariement optimal pour les disposantes : {'h1': 'f2', 'h2': 'f1', 'h3': 'f4', 'h4': 'f3'}


In [7]:
// Verification des deux promesses de Gale-Shapley, sur les deux appariements.

fn prefs_de<'a>(cote: &[(&'a str, [&'a str; 4])], nom: &str) -> [&'a str; 4] {
    cote.iter().find(|(n, _)| *n == nom).map(|(_, p)| *p).unwrap()
}

fn conjoint_de<'a>(m: &[(&'a str, &'a str)], p: &str) -> &'a str {
    m.iter().find(|(pp, _)| *pp == p).map(|(_, d)| *d).unwrap()
}

fn partenaire_de<'a>(m: &[(&'a str, &'a str)], d: &str) -> &'a str {
    m.iter().find(|(_, dd)| *dd == d).map(|(p, _)| *p).unwrap()
}

fn py_tuple_ss(p: &str, d: &str) -> String {
    format!("('{}', '{}')", p, d)
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

// Deferred acceptance : les proposants demandent, les disposantes choisissent.
fn gale_shapley<'a>(
    proposants: &[(&'a str, [&'a str; 4])],
    disposantes: &[(&'a str, [&'a str; 4])],
) -> (Vec<(&'a str, &'a str)>, usize) {
    let mut libres: std::collections::VecDeque<usize> = (0..proposants.len()).collect();
    let mut prochain_choix = vec![0usize; proposants.len()];
    let mut fiance: Vec<Option<usize>> = vec![None; disposantes.len()];
    let mut demandes = 0usize;
    while let Some(pi) = libres.pop_front() {
        let d = proposants[pi].1[prochain_choix[pi]];
        prochain_choix[pi] += 1;
        demandes += 1;
        let di = disposantes.iter().position(|(n, _)| *n == d).unwrap();
        match fiance[di] {
            None => fiance[di] = Some(pi),
            Some(cur) => {
                let prefs = &disposantes[di].1;
                if prefs.iter().position(|h| *h == proposants[pi].0).unwrap()
                    < prefs.iter().position(|h| *h == proposants[cur].0).unwrap()
                {
                    libres.push_back(cur);
                    fiance[di] = Some(pi);
                } else {
                    libres.push_back(pi);
                }
            }
        }
    }
    let paires = fiance
        .iter()
        .enumerate()
        .map(|(di, opt)| (proposants[opt.unwrap()].0, disposantes[di].0))
        .collect();
    (paires, demandes)
}

// Cherche une paire (p, d) qui prefererait mutuellement se quitter.
fn paire_bloquante(
    m: &[(&str, &str)],
    proposants: &[(&str, [&str; 4])],
    disposantes: &[(&str, [&str; 4])],
) -> Option<(String, String)> {
    for (p, prefs_p) in proposants.iter() {
        for (d, prefs_d) in disposantes.iter() {
            if *d == conjoint_de(m, p) {
                continue;
            }
            let actuel_p = conjoint_de(m, p);
            let actuel_d = partenaire_de(m, d);
            if prefs_p.iter().position(|x| *x == *d).unwrap() < prefs_p.iter().position(|x| *x == actuel_p).unwrap()
                && prefs_d.iter().position(|x| *x == *p).unwrap() < prefs_d.iter().position(|x| *x == actuel_d).unwrap()
            {
                return Some((p.to_string(), d.to_string()));
            }
        }
    }
    None
}

fn rang(pref: &[&str], x: &str) -> usize {
    pref.iter().position(|e| *e == x).unwrap() + 1 // rang 1 = meilleur
}

{
    let proposants: [(&str, [&str; 4]); 4] = [
        ("h1", ["f1", "f2", "f3", "f4"]),
        ("h2", ["f2", "f1", "f4", "f3"]),
        ("h3", ["f3", "f4", "f1", "f2"]),
        ("h4", ["f4", "f3", "f2", "f1"]),
    ];
    let disposantes: [(&str, [&str; 4]); 4] = [
        ("f1", ["h2", "h1", "h4", "h3"]),
        ("f2", ["h1", "h2", "h3", "h4"]),
        ("f3", ["h4", "h3", "h2", "h1"]),
        ("f4", ["h3", "h4", "h1", "h2"]),
    ];
    let (m_hommes, _) = gale_shapley(&proposants, &disposantes);
    let (m_femmes_brut, _) = gale_shapley(&disposantes, &proposants);
    let m_femmes: Vec<(&str, &str)> = m_femmes_brut.iter().map(|(f, h)| (*h, *f)).collect();

    for (nom, m) in [("optimal proposants", &m_hommes), ("optimal disposantes", &m_femmes)] {
        let bloq = paire_bloquante(m, &proposants, &disposantes);
        let verdict = match &bloq {
            None => "OUI".to_string(),
            Some((p, d)) => format!("NON, paire bloquante {}", py_tuple_ss(p, d)),
        };
        println!("{:<20} stable : {}", nom, verdict);
    }
    println!();
    println!("Rangs obtenus par les proposants (1 = meilleur) :");
    println!("{:<18} {:>4} {:>4} {:>4} {:>4}", "", "h1", "h2", "h3", "h4");
    let rh: Vec<usize> = ["h1", "h2", "h3", "h4"]
        .iter()
        .map(|h| rang(&prefs_de(&proposants, h), conjoint_de(&m_hommes, h)))
        .collect();
    let rf: Vec<usize> = ["h1", "h2", "h3", "h4"]
        .iter()
        .map(|h| rang(&prefs_de(&proposants, h), conjoint_de(&m_femmes, h)))
        .collect();
    println!("{:<18} {:>4} {:>4} {:>4} {:>4}", "opt. proposants", rh[0], rh[1], rh[2], rh[3]);
    println!("{:<18} {:>4} {:>4} {:>4} {:>4}", "opt. disposantes", rf[0], rf[1], rf[2], rf[3]);
    println!();
    println!(
        "Chaque proposant est au moins aussi bien place dans l'appariement qu'il initie : {}",
        py_bool(rh.iter().zip(rf.iter()).all(|(a, b)| a <= b))
    );
};

optimal proposants   stable : OUI


optimal disposantes  stable : OUI


Rangs obtenus par les proposants (1 = meilleur) :


                     h1   h2   h3   h4


opt. proposants       1    1    1    1


opt. disposantes      2    2    2    2


Chaque proposant est au moins aussi bien place dans l'appariement qu'il initie : True


**Lecture du résultat.** Les deux appariements sont **stables** (aucune paire bloquante — la vérification tourne sur toutes les pires paires possibles), mais ils **diffèrent** : dans celui que les proposants initient, chaque proposant est au rang qu'il a dans l'autre ou mieux. La stabilité n'est pas neutre — **celui qui demande est favorisé**. C'est exactement pourquoi le mécanisme d'admission aux internats américains a changé de côté en 1998 : laisser les étudiants proposer (et non les hôpitaux) leur donnait l'optimalité sans perdre la stabilité.

SC-08 pousse chaque affirmation jusqu'à la preuve : couplage parfait garanti, man-optimalité **vérifiée par énumération exhaustive** de tous les appariements stables, et — côté formel — le lake Lean `game_theory_lean/StableMarriage/` prouve la correction de l'algorithme sans un seul `sorry`.


## 3. Ce que le lake Lean prouve (et que la simulation ne prouve pas)

La sous-série porte un **lake Lean 4** de référence : le dossier `SocialChoice/social_choice_core_lean/SocialChoice/`, construit sur les notes de Dominik Peters. Les cellules suivantes le lisent **dans le dépôt** — ce qu'elles affichent est mesuré, pas recopié d'une documentation.


In [8]:
// Lecture directe du lake : enonce d'Arrow tel que prouve, et inventaire mesure.

{
    let lake = std::path::Path::new("SocialChoice").join("social_choice_core_lean").join("SocialChoice");
    let fleche = std::fs::read_to_string(lake.join("Arrow.lean"))
        .expect("Arrow.lean illisible (noyau lance depuis le dossier du carnet)");

    // Extraire la docstring du theoreme arrow et sa signature (par index : un motif
    // /--.*-/ non ancre avalerait toutes les defs intermediaires du fichier).
    let i = fleche.find("theorem arrow").expect("theorem arrow introuvable dans Arrow.lean");
    let j = fleche[..i].rfind("/--").expect("docstring introuvable"); // docstring immediatement precedente
    let k = j + fleche[j..].find("-/").expect("fin de docstring introuvable");
    let docstring = fleche[j + 3..k].trim();
    let signature = &fleche[i..i + fleche[i..].find(" := ").expect("signature non terminee")];

    println!("Enonce tel qu'il vit dans Arrow.lean :");
    println!("  {}", docstring.lines().map(|l| l.trim()).collect::<Vec<&str>>().join("\n"));
    println!();
    println!("Signature Lean :");
    for l in signature.lines() {
        println!("  {}", l);
    }
};

Enonce tel qu'il vit dans Arrow.lean :


  Théorème d'impossibilité d'Arrow :


Toute fonction de bien-être social sur au moins 3 alternatives qui satisfait


le Pareto faible et l'indépendance des alternatives non pertinentes doit


être une dictature.


Preuve alternative déclarée : `lean_game_defs/SocialChoice.lean` (maison


... (3 outputs left out)

Signature Lean :


  theorem arrow (f : SWF ι σ) (X : Finset σ)


      (hwp : weak_pareto f X) (hind : ind_of_irr_alts f X) (hX : 3 ≤ X.card) :


      is_dictatorship f X


In [9]:
// Inventaire mesure du lake : modules, theoremes/lemmes, sorry.

{
    // Le chemin du lake est relatif au dossier du carnet, comme la cellule precedente.
    let lake = std::path::Path::new("SocialChoice").join("social_choice_core_lean").join("SocialChoice");
    let mut modules: Vec<String> = Vec::new();
    for entry in std::fs::read_dir(&lake).expect("lake SocialChoice illisible") {
        let entry = entry.expect("entree du lake illisible");
        let nom = entry.file_name().to_string_lossy().to_string();
        if nom.ends_with(".lean") && !nom.ends_with("_en.lean") && !nom.starts_with('_') {
            modules.push(nom);
        }
    }
    modules.sort();

    let mut total_thm = 0usize;
    let mut total_sorry = 0usize;
    for module in &modules {
        let texte = std::fs::read_to_string(lake.join(module)).expect("module du lake illisible");
        let thm = texte
            .lines()
            .filter(|l| l.starts_with("theorem ") || l.starts_with("lemma "))
            .count();
        let sorry = texte.matches("sorry").count();
        total_thm += thm;
        total_sorry += sorry;
    }

    let sans_ext: Vec<String> = modules.iter().map(|m| m.replace(".lean", "")).collect();
    println!("{} modules (plus leurs miroirs _en) : {}", modules.len(), sans_ext.join(", "));
    println!("{} theoremes/lemmes, {} 'sorry'.", total_thm, total_sorry);
    println!("La CI (.github/workflows/lean-ci-matrix.yml, lake social_choice_core_lean de scripts/lean/ci_lakes.json) certifie ce zero-sorry a chaque push.");
};

10 modules (plus leurs miroirs _en) : AMD, Arrow, Basic, Framework, GibbardSatterthwaite, Groves, MechanismDesign, Sen, SortedListCounting, Voting


129 theoremes/lemmes, 0 'sorry'.


La CI (.github/workflows/lean-ci-matrix.yml, lake social_choice_core_lean de scripts/lean/ci_lakes.json) certifie ce zero-sorry a chaque push.


**Lecture du résultat.** L'énoncé que la simulation a *effleuré* sur un profil, le lake le prouve **pour toute fonction de bien-être social sur au moins 3 alternatives** — l'infinité de cas que ni la simulation (elle échantillonne), ni SAT (il vérifie les cas finis, SC-04) ne couvrent. C'est la gradation complète de la sous-série : **voir** (simuler, SC-01), **vérifier** (SAT, SC-04), **prouver** (Lean, SC-02). Le capstone s'arrête à la première marche ; les deux autres sont le parcours Recherche ci-dessous.


## 4. Par où y entrer

Trois parcours de lecture dans la sous-série, selon ce que vous cherchez :

- **Découverte** (~1 h) : [SC-03 Méthodes de vote](SocialChoice/ChoixSocial-03-Voting-Methods-Python.ipynb) puis [SC-05 Gibbard-Satterthwaite](SocialChoice/ChoixSocial-05-Gibbard-Satterthwaite-Python.ipynb). Aucun prérequis au-delà de ce capstone : les paradoxes électoraux d'abord, la manipulation ensuite.
- **Licence** (~2 h) : [SC-01 Arrow par la simulation](SocialChoice/ChoixSocial-01-Arrow-Impossibility-Theorem-Python.ipynb) (la preuve de Geanakoplos déroulée par le code), puis [SC-04 SAT/Z3](SocialChoice/ChoixSocial-04-Computational-Aggregation-SAT-Z3-Python.ipynb) (l'impossibilité devenue `UNSAT`), puis [SC-08 Gale-Shapley](SocialChoice/ChoixSocial-02-Stable-Marriage-Gale-Shapley-Rust.ipynb) (la face constructive).
- **Recherche** (~4 h) : [SC-02 Choix social formel en Lean](SocialChoice/ChoixSocial-01b-Lean-SocialChoice-Formal-Lean.ipynb) (Arrow, Sen, électeur médian, tour du lake de D. Peters), [SC-06 Möbius et pouvoir](SocialChoice/ChoixSocial-06-Mobius-Aggregation-Pouvoir-Manipulation-Rust.ipynb), [SC-07 Comités et core](SocialChoice/ChoixSocial-07-Committees-Core-Python.ipynb).

Le [README de la sous-série](SocialChoice/README.md) porte le parcours complet, les durées, et le diagramme des dépendances entre les huit angles.


## 5. Exercices

Trois exercices pour passer de lecteur à praticien. Chaque stub s'exécute (il rend `None` ou affiche un message) — complétez-le, la cellule de vérification qui suit reste valable.


In [10]:
// Exercice 1 -- Copeland sur le profil cyclique.
// Score de Copeland : +1 par victoire en duel majoritaire, +0.5 par egalite, 0 par defaite.
// Indice : reutilisez la logique de `duels` ci-dessus ; le profil CYCLE est defini en 2.2.
// Etape 1 : pour chaque alternative, compter victoires et egalites.
// Etape 2 : retourner le dict {alternative: score}.

fn copeland(_profil: &[Vec<&str>], _alts: &[&str]) -> Option<Vec<(String, f64)>> {
    // TODO etudiant
    None
}

{
    println!("Exercice a completer : copeland(CYCLE, ALTS) doit rendre un dict -- que remarquez-vous sur le profil cyclique ?");
};

Exercice a completer : copeland(CYCLE, ALTS) doit rendre un dict -- que remarquez-vous sur le profil cyclique ?


In [11]:
// Exercice 2 -- Trouver une manipulation de Borda.
// Avec le profil PROFIL_ARROW (2.1) et l'electeur 2 (indice 1), cherchez un bulletin
// NON sincere (un ordre different de ses preferences reelles) qui ameliore son issue,
// les autres bulletins restant fixes.
// Indice : enumeratez les 6 ordres possibles pour l'electeur 2 ; comparez le gagnant.
// Etape 1 : gagnant sincere = argmax borda(PROFIL_ARROW).
// Etape 2 : pour chaque ordre alternatif, gagnant hypothetique ; garder la premiere amelioration
//           pour l'electeur 2 (l'ordre reel de l'electeur 2 est PROFIL_ARROW[1]).

fn manipule_borda(_profil: &[Vec<&str>], _alts: &[&str], _indice_electeur: usize) -> Option<Vec<String>> {
    // TODO etudiant
    None
}

{
    println!("Exercice a completer : manipule_borda(PROFIL_ARROW, ALTS, 1) doit rendre un ordre non sincere qui ameliore l'electeur 2, ou None s'il n'en existe pas.");
};

Exercice a completer : manipule_borda(PROFIL_ARROW, ALTS, 1) doit rendre un ordre non sincere qui ameliore l'electeur 2, ou None s'il n'en existe pas.


In [12]:
// Exercice 3 -- Une paire bloquante a la main.
// On retire h1 de l'appariement optimal-proposants et on le fiance a f4 par decree.
// Cette paire (h1, f4) est-elle bloquante ? Et lesquels des deux ex-fiances veulent fuir ?
// Indice : comparez, pour h1 ET pour f4, le rang du partenaire impose a celui qu'ils avaient.
// Etape 1 : appariement modifie M2 = copie de M_hommes avec la permutation.
// Etape 2 : appliquez paire_bloquante(M2, ...) et interpretez la paire rendue.

fn appariement_modifie() -> Option<(String, String)> {
    // TODO etudiant
    None
}

{
    println!("Exercice a completer : construisez M2 et rendez la paire bloquante rendue par paire_bloquante(M2, PROPOSANTS, DISPOSANTES).");
};

Exercice a completer : construisez M2 et rendez la paire bloquante rendue par paire_bloquante(M2, PROPOSANTS, DISPOSANTES).


## Conclusion

Ce capstone a refermé une boucle : le parcours GameTheory vous a appris à **chercher un équilibre** (que feront des joueurs rationnels ?) ; la sous-série SocialChoice demande la question duale — **comment décider ensemble** quand les préférences divergent ? — et y répond par des théorèmes d'impossibilité dressés par des preuves, des solveurs et des simulations.

- Pour suivre le fil complet : [README de la sous-série SocialChoice](SocialChoice/README.md) ;
- Pour revenir au parcours : le [palier 16](GameTheory-16-MechanismDesign-Python.ipynb) (mécanismes et révélation) est le point d'ancrage naturel ; les [extensions au-delà du parcours](README.md#extensions-au-delà-du-parcours) poursuivent dans une autre direction ;
- Le lake formel : `SocialChoice/social_choice_core_lean/SocialChoice/` (certifié sans `sorry` par la CI), visité en détail par [SC-02](SocialChoice/ChoixSocial-01b-Lean-SocialChoice-Formal-Lean.ipynb).

**Ce que ce notebook suppose** était en tête de carnet ; ce qu'il laisse ouvert est dans les exercices — le capstone se referme sur eux.
